In [ ]:
#!pip --version

pip 26.2.1 from D:\in0902\ex0914\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
from dotenv import load_dotenv


load_dotenv()

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel , Field

from itertools import chain
from langchain_core.prompts import PromptTemplate

from langchain_teddynote.messages import stream_response

In [4]:
llm = ChatOpenAI(temperature=0, model="gpt-4.1-mini")

In [5]:
#이메일 예시
email_conversation = """From : 김철수 (chulsoo.kim@bikecorporation.me)
To : 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,
저는 바이크코퍼레이션 김철수 상무입니다. 최근 보도자료를 통해 귀사의 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크 코퍼레이션은~
ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보~

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [6]:
#출력 파서를 사용 X 경우
Prompt = PromptTemplate.from_template(
    "다음의 이메일 내용중 중요한 내용을 추출해 주세요. \n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0, model="gpt-4.1-mini")

chain = Prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

중요 내용 요약:

- 발신자: 김철수 상무 (바이크코퍼레이션)
- 수신자: 이은채 대리 (테디인터내셔널)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
- 주요 요청 사항: ZENESIS 모델에 대한 상세 브로슈어 요청
  - 특히 기술 사양, 배터리 성능, 디자인 정보 포함
- 목적: 자전거 유통 협력 논의를 위한 정보 확보 및 미팅 일정 조율

In [7]:
output

'중요 내용 요약:\n\n- 발신자: 김철수 상무 (바이크코퍼레이션)\n- 수신자: 이은채 대리 (테디인터내셔널)\n- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안\n- 주요 요청 사항: ZENESIS 모델에 대한 상세 브로슈어 요청\n  - 특히 기술 사양, 배터리 성능, 디자인 정보 포함\n- 목적: 자전거 유통 협력 논의를 위한 정보 확보 및 미팅 일정 조율'

In [9]:
class EmailSummary(BaseModel):
    person:str = Field(description="메일을 보낸 사람")
    email:str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject:str = Field(description="메일 제목")
    summary:str = Field(description="메일 본문을 요약한 텍스트")
    date:str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")
    

In [10]:
#pydanticOutParser 생성
parser = PydanticOutputParser(pydantic_object=EmailSummary)
parser

PydanticOutputParser(pydantic_object=<class '__main__.EmailSummary'>)

In [11]:
#instruction 을 출력
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [13]:
#프롬프트 탬플릿
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT
{format}
"""
)

In [15]:
#format 에 PydanticOutParser 의 부분 포맷됨(partial)추가
prompt = prompt.partial(format=parser.get_format_instructions())
prompt

PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [18]:
# chain생성
chain = prompt | llm

# chain실행 및 결과 출력
response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 JSON 형태로 출력
output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거 모델에 대한 상세한 브로슈어 요청과 유통 협력 및 미팅 일정을 제안함. 특히 기술 사양, 배터리 성능, 디자인 정보에 대한 자료를 요청함.",
  "date": ""
}
```

In [19]:
#PydanticOutParser 결과 파싱
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary='바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거 모델에 대한 상세한 브로슈어 요청과 유통 협력 및 미팅 일정을 제안함. 특히 기술 사양, 배터리 성능, 디자인 정보에 대한 자료를 요청함.' date=''


In [20]:
structured_output

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거 모델에 대한 상세한 브로슈어 요청과 유통 협력 및 미팅 일정을 제안함. 특히 기술 사양, 배터리 성능, 디자인 정보에 대한 자료를 요청함.', date='')

In [21]:
structured_output.person

'김철수'

In [22]:
structured_output.email

'chulsoo.kim@bikecorporation.me'

In [23]:
structured_output.date

''

In [24]:
structured_output.subject

'"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안'

In [29]:
# : 프롬프트 주입 방식을 사용하는 . chain 재구성
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary='바이크코퍼레이션 김철수 상무가 이은채 대리에게 ZENESIS 자전거 모델에 대한 상세 브로슈어 요청과 유통 협력 및 미팅 일정을 제안함. 특히 기술 사양, 배터리 성능, 디자인 정보에 대한 자료를 요청함.', date='')